In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

FILE = "Water Quality.csv"  # Change to full path if needed
OUT = "task1_domain_features_outputs"
ALIASES = {
    "pH": ["ph"], "temperature": ["temperature", "water temperature"],
    "conductivity": ["specific conductance", "conductivity", "electrical conductivity"],
    "dissolved_oxygen": ["dissolved oxygen"],
    "oxidation_reduction_potential": ["oxidation reduction potential", "redox potential", "orp"],
    "alkalinity": ["alkalinity", "total alkalinity"], "hardness": ["hardness", "total hardness"],
    "calcium": ["calcium"], "magnesium": ["magnesium"], "sodium": ["sodium"], "potassium": ["potassium"],
    "chloride": ["chloride"], "sulfate": ["sulfate", "sulphate"],
    "bicarbonate": ["bicarbonate", "hydrogen carbonate"], "carbonate": ["carbonate"],
    "nitrate": ["nitrate"], "phosphate": ["phosphate", "orthophosphate"],
    "ammonium": ["ammonium"], "iron": ["iron"], "manganese": ["manganese"]
}
MEQ = {"calcium":2/40.078, "magnesium":2/24.305, "sodium":1/22.989769,
       "potassium":1/39.0983, "chloride":1/35.453, "sulfate":2/96.06,
       "bicarbonate":1/61.0168, "carbonate":2/60.009, "nitrate":1/62.0049,
       "phosphate":3/94.9714}

def canon(v):
    s = str(v).strip().lower()
    for name, aliases in ALIASES.items():
        if s in [a.lower() for a in aliases]:
            return name
    return None

def main():
    if not Path(FILE).exists():
        raise FileNotFoundError(f"Cannot find {FILE!r}; set FILE to the full CSV path.")
    Path(OUT).mkdir(exist_ok=True)
    df = pd.read_csv(FILE, low_memory=False)
    req = {"CharacteristicName","ResultValue"}
    if req-set(df.columns): raise ValueError(f"Missing columns: {req-set(df.columns)}")
    df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
    df = df.dropna(subset=["CharacteristicName","ResultValue"]).copy()
    if "ActivityStartDate" in df:
        df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    df["canonical_parameter"] = df["CharacteristicName"].map(canon)
    matched = df.dropna(subset=["canonical_parameter"]).copy()
    if matched.empty:
        df[["CharacteristicName"]].drop_duplicates().to_csv(Path(OUT)/"available_characteristic_names.csv", index=False)
        print("No aliases matched. See available_characteristic_names.csv and edit ALIASES.")
        return
    group = [c for c in ["MonitoringLocationID","ActivityStartDate","ResultUnit"] if c in matched.columns]
    if "ResultUnit" in matched: matched["ResultUnit"] = matched["ResultUnit"].fillna("Unknown").astype(str)
    wide = matched.groupby(group+["canonical_parameter"], dropna=False).ResultValue.median().reset_index()
    wide = wide.pivot_table(index=group, columns="canonical_parameter", values="ResultValue", aggfunc="median").reset_index()
    wide.columns.name = None
    unit_ok = "ResultUnit" in wide and wide["ResultUnit"].str.lower().str.replace(" ","").eq("mg/l").all()
    ions = ["calcium","magnesium","sodium","potassium","chloride","sulfate","bicarbonate"]
    if unit_ok and all(c in wide for c in ions):
        cat = sum(wide[c]*MEQ[c] for c in ["calcium","magnesium","sodium","potassium"])
        ani = sum(wide[c]*MEQ[c] for c in ["chloride","sulfate","bicarbonate"])
        wide["cation_sum_meq_L"], wide["anion_sum_meq_L"] = cat, ani
        wide["ion_balance_error_percent"] = np.where(cat+ani != 0, 100*(cat-ani)/(cat+ani), np.nan)
    else:
        print("Ion balance not calculated: needs Ca, Mg, Na, K, Cl, sulfate, bicarbonate all in mg/L with matching sample/time.")
    if unit_ok and "calcium" in wide and "magnesium" in wide:
        wide["Ca_to_Mg_molar_ratio"] = (wide["calcium"]/40.078)/(wide["magnesium"]/24.305).replace(0,np.nan)
    wide.to_csv(Path(OUT)/"domain_feature_table.csv", index=False)
    print("Saved domain feature table to", OUT)
    print("ORP is retained only if measured; it is not inferred from other columns.")
    print("Review whether location/month aggregation aligns the same physical sample before interpreting ion balance.")
if __name__ == "__main__": main()


Ion balance not calculated: needs Ca, Mg, Na, K, Cl, sulfate, bicarbonate all in mg/L with matching sample/time.
Saved domain feature table to task1_domain_features_outputs
ORP is retained only if measured; it is not inferred from other columns.
Review whether location/month aggregation aligns the same physical sample before interpreting ion balance.
